# Data Preparation

## Preparation Objective

This stage aims to produce an auditable, respondent-aligned, fully documented dataset and algorithm-specific inputs for the subsequent clustering stage.

The notebook uses the `src.data_preparation` package, including its current schema mapping, cleaning rules, type harmonisation, missingness registers, feature decisions, transformations, validation, and persistence functions. It preserves the original survey responses, distinguishes missing ness from confirmed non-applicability, and keeps descriptive variables separate from the primary full-population clustering candidates.

### 1. Purpose Context and Preparation Design

The source contains 1,433 respondent records and 63 survey fields. The preparation design retains all respondents and preserves the distinction between:
- **Observed responses:** values actually supplied by respondents.
- **Item nonresponse:** no repsonse recorded for a field.
- **Structural non-applicability:** a field was not applicable because of the survey's routing or respondent circumstances.
- **Derived representation:** additional columns created for analysis while retaining the source response.

The principal representations are K-Modes categorical input, MCA categorical input and coordinates, and a Gower mixed-data input and distance matrix. UMAP is derived from MCA coordinates as a two-dimensional exploratory representation; it is not treated as a clustering result or as evidence of cluster quality.

### 2. Environment and Explicit Package Imports



In [5]:
from pathlib import Path
import sys

# Dynamically find the project root by walking up parent directories until we find 'src'
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir
for parent in [current_dir] + list(current_dir.parents):
    if (parent / "src").is_dir():
        PROJECT_ROOT = parent
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_preparation.config import (
    PROJECT_ROOT,
    RAW_DIR,
    PROCESSED_DIR,
    SOURCE_FILENAME,
    SOURCE_PATH,
    RANDOM_STATE,
    EXPECTED_RAW_SHAPE,
    INVALID_AGES,
)
from src.data_preparation.ingestion import (
    load_source,
    sha256_file,
)
from src.data_preparation.schema import (
    rename_columns,
    build_column_mapping,
)
from src.data_preparation.cleaning import (
    normalize_string_columns,
    clean_age,
    clean_gender,
    clean_countries,
    normalize_multiselect_columns,
    normalize_text_value,
    normalize_gender,
)
from src.data_preparation.type_harmonization import (
    canonicalise_response,
    canonicalise_yes_no,
    harmonise_categorical_columns,
    build_type_metadata,
)
from src.data_preparation.missingness import (
    STRUCTURALLY_CONDITIONAL_FIELDS,
    FULL_POPULATION_EXCLUSIONS,
    build_missingness_audit,
    build_applicability_register,
    build_exclusion_register,
)
from src.data_preparation.feature_engineering import (
    MULTISELECT_COLUMNS,
    COMPANY_SIZE_ORDER,
    REMOTE_WORK_ORDER,
    parse_multiselect_columns,
    encode_multiselect,
    encode_work_position,
    add_ordinal_company_size,
    add_remote_work_ordinal,
)
from src.data_preparation.feature_selection import (
    CORE_CATEGORICAL_FEATURES,
    CONDITIONAL_FEATURE_CANDIDATES,
    EXCLUSION_REASONS,
    build_feature_decision_register,
    select_core_features,
    assert_expected_features_present,
)
from src.data_preparation.transformations import (
    MISSING_CATEGORY,
    apply_structural_applicability,
    prepare_kmodes_matrix,
    prepare_mca_input,
    fit_mca_coordinates,
    prepare_gower_input,
    gower_distance_matrix,
    prepare_umap_input,
    fit_umap_embedding,
)
from src.data_preparation.validation import (
    validate_row_lineage,
    validate_categorical_matrix,
    validate_numeric_matrix,
    validate_gower_matrix,
    validate_model_input_bundle,
)
from src.data_preparation.persistence import (
    ensure_output_directory,
    save_dataframe,
    save_numpy,
    save_json,
    save_preparation_outputs,
    build_respondent_row_mapping,
    build_feature_metadata,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

warnings.filterwarnings(
    "default",
    category=FutureWarning,
)

print(f"Python: {sys.version.split()[0]}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Source path configured by package: {SOURCE_PATH}")
print(f"Expected raw shape: {EXPECTED_RAW_SHAPE}")
print(f"Random state: {RANDOM_STATE}")

Python: 3.13.7
Project root: C:\Users\User\Documents\Projects\osmi-mental-health-data-clustering
Source path configured by package: C:\Users\User\Documents\Projects\osmi-mental-health-data-clustering\data\raw\mental-heath-in-tech-2016_20161114.csv
Expected raw shape: (1433, 63)
Random state: 42


### 3. Source Ingestion and Integrity

The ingestion function enforces the expected source dimensions and records a SHA-256 digest. This makes the provenance of the prepared data traceable to a specific source file. The original CSV remains immutable: all subsequent operations work on derived copies.

In [11]:
# The project source file is expected in data/raw/.
# The uploaded source file may be copied there without modifying its contents.

raw_df, source_metadata = load_source(SOURCE_PATH)

display(
    pd.Series(
        source_metadata,
        name="Source metadata",
    ).to_frame()
)

assert raw_df.shape == EXPECTED_RAW_SHAPE
assert raw_df.columns.is_unique
assert raw_df.index.is_unique

print(f"Source shape: {raw_df.shape}")

,Source metadata
source_path,C:\Users\User\Documents\Projects\osmi-mental-health-data-clustering\data\raw\mental-heath-in-tech-2016_20161114.csv
source_filename,mental-heath-in-tech-2016_20161114.csv
source_sha256,0bec458b0724cc375a17eb2db0204a9f7a786260441cf702eec210d92bd4ae4d
rows,1433
columns,63
duplicate_rows,0
duplicate_columns,0


Source shape: (1433, 63)


#### 3.1 Source-Level Quality Profile

In [13]:
source_profile = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing_count": raw_df.isna().sum(),
    "missing_pct": raw_df.isna().mean().mul(100).round(2),
    "distinct_non_missing": raw_df.nunique(dropna=True),
})

source_profile = source_profile.sort_values(
    ["missing_pct", "distinct_non_missing"],
    ascending=[False, False],
)

display(source_profile)

total_cells = raw_df.shape[0] * raw_df.shape[1]
total_missing = int(raw_df.isna().sum().sum())

print(f"Total cells: {total_cells:,}")
print(f"Missing cells: {total_missing:,}")
print(f"Overall missingness: {total_missing / total_cells:.2%}")

,dtype,missing_count,missing_pct,distinct_non_missing
"If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?",str,1289,89.95,3
"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",str,1229,85.76,4
Is your primary role within your company related to tech/IT?,float64,1170,81.65,2
"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?",str,1146,79.97,5
"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?",str,1146,79.97,5
"If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?",str,1146,79.97,4
Do you believe your productivity is ever affected by a mental health issue?,str,1146,79.97,4
Do you know local or online resources to seek help for a mental health disorder?,str,1146,79.97,3
Do you have medical coverage (private insurance or state-provided) which includes treatment of mental health issues?,float64,1146,79.97,2
"If maybe, what condition(s) do you believe you have?",str,1111,77.53,99


Total cells: 90,279
Missing cells: 21,960
Overall missingness: 24.32%


The overall missing-cell percentage is a dataset-wide summary, not a measure of respondent completeness or a justification for imputation. Survey routing means that some unanswered fields are expected for particular respondents. The field-level missingness audit and applicability register are therefore more informative than a single aggregate percentage.

#### 3.2 Duplicate Respondent Assessment

In [15]:
duplicate_row_count = int(raw_df.duplicated().sum())

duplicate_report = pd.DataFrame({
    "measure": [
        "Rows",
        "Exact duplicate rows",
        "Unique rows",
        "Duplicate source column labels",
    ],
    "value": [
        len(raw_df),
        duplicate_row_count,
        raw_df.drop_duplicates().shape[0],
        int(raw_df.columns.duplicated().sum()),
    ],
})

display(duplicate_report)

assert duplicate_row_count == 0

,measure,value
0,Rows,1433
1,Exact duplicate rows,0
2,Unique rows,1433
3,Duplicate source column labels,0


No exact duplicate respondent rows are present in the supplied source. No respondent is removed on the basis of duplicate detection.

### 4. Canonical Schema and Respondent Lineage

The source questionnaire uses long natural-language field labels. The package's `rename_columns` function maps these labels to stable snake-case names and returns a separate mapping audit. The source row identifier is retained independently of the DataFrame's working index.

In [22]:
renamed_df, column_mapping_audit = rename_columns(raw_df)

renamed_df.insert(
    0,
    "source_row_id",
    np.arange(len(renamed_df), dtype=np.int64),
)

renamed_df.insert(
    1,
    "source_index",
    raw_df.index.to_numpy(),
)

display(column_mapping_audit)

assert renamed_df.shape[0] == raw_df.shape[0]
assert renamed_df.columns.is_unique
assert renamed_df["source_row_id"].is_unique
assert renamed_df["source_index"].is_unique

print(f"Canonicalised data shape: {renamed_df.shape}")
print(f"Canonical field count: {len(column_mapping_audit)}")

,original_name,canonical_name
0,Are you self-employed?,self_employed
1,How many employees does your company or organization have?,company_size
2,Is your employer primarily a tech company/organization?,tech_employer
3,Is your primary role within your company related to tech/IT?,tech_role
4,Does your employer provide mental health benefits as part of healthcare coverage?,employer_mental_health_benefits
5,Do you know the options for mental health care available under your employer-provided coverage?,employer_benefits_awareness
6,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other offici...",employer_mental_health_discussion
7,Does your employer offer resources to learn more about mental health concerns and options for seeking help?,employer_mental_health_resources
8,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources ...,employer_confidentiality
9,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",medical_leave_ease


Canonicalised data shape: (1433, 65)
Canonical field count: 63


#### 4.1 Schema Mapping Review

In [24]:
unmapped_headers = column_mapping_audit[
    column_mapping_audit["canonical_name"].str.contains(
        r"^unnamed_column$",
        regex=True,
    )
]

canonical_name_collisions = (
    column_mapping_audit["canonical_name"]
    .duplicated(keep=False)
)

print(f"Unmapped/unnamed canonical headers: {len(unmapped_headers)}")
print(f"Canonical-name collisions: {int(canonical_name_collisions.sum())}")

display(unmapped_headers)

assert not canonical_name_collisions.any()

Unmapped/unnamed canonical headers: 0
Canonical-name collisions: 0


,original_name,canonical_name


The mapping explicitly handles the two separate "Why or why not?" questionnaire items through their parser-generated source labels. It also normalises Unicode whitespace in headers, including the non-breaking space present in the medical-coverage question.

### 5. Text Normalisation and Missing-Value Representation

Whitespace, Unicode compatibility forms, non-breaking spaces, zero-width characters, and non-printing control characters can create artificial categories. The package's `normalize_string_columns` function standardises text formatting and converts empty strings to missing values. It does not consolidate semantically different answers.

In [33]:
cleaned_df = normalize_string_columns(renamed_df)

assert cleaned_df["source_row_id"].equals(renamed_df["source_row_id"])
assert cleaned_df.shape[0] == renamed_df.shape[0]

normalisation_comparison = pd.DataFrame({
    "before_missing": renamed_df.isna().sum(),
    "after_missing": cleaned_df.isna().sum(),
})

normalisation_comparison["new_missing"] = (
    normalisation_comparison["after_missing"]
    - normalisation_comparison["before_missing"]
)

display(
    normalisation_comparison.sort_values(
        "new_missing",
        ascending=False,
    ).query("new_missing != 0")
)

print(
    "Missing cells after text normalisation:",
    f"{cleaned_df.isna().sum().sum():,}",
)

,before_missing,after_missing,new_missing


Missing cells after text normalisation: 21,960
